<div dir="rtl" align="right">

# نطاقاتُ الترددِ في إشارةِ EEG

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1 (مُشاركٌ واحدٌ فقط للسرعةِ)

---

## نظرةٌ عامّةٌ

تُصنّفُ إشارةُ EEG في خمسةِ نطاقاتٍ تردديّةٍ رئيسةٍ، يَرتبطُ كلٌّ منها بحالةٍ دماغيّةٍ مُحدّدة. يُطبّقُ هذا الدفترُ تحليلَ الطيفِ  (تحويلَ فورييه السريعَ) على إشارةٍ حقيقيّةٍ ويُبرزُ النطاقاتِ الموجودةَ فيها.

| النطاقُ | الترددُ | الحالةُ الدماغيّةُ |
| ------- | ------- | --------- |
| دلتا | 0.5 إلى 4 Hz | النومُ العميقُ |
| ثيتا | 4 إلى 8 Hz | النعاسُ، الذاكرةُ |
| ألفا | 8 إلى 13 Hz | الاسترخاءُ (إغلاقُ العينينِ) |
| بيتا | 13 إلى 30 Hz | التفكيرُ النشطُ، التركيزُ |
| غاما | 30 إلى 100 Hz | المعالجةُ المعرفيّةُ العليا |

## المُخرجاتُ المُتوقّعةُ

رسمٌ بيانيٌّ يُظهرُ توزيعَ الطاقةِ على التردداتِ من 0 إلى 100 Hz. تَتركّزُ الطاقةُ في النطاقاتِ المنخفضةِ (دلتا وثيتا وألفا) وتَتناقصُ في النطاقاتِ الأعلى، وفقاً للقانونِ الطيفيِّ $1/f$.

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb

<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1

<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')

<div dir="rtl" align="right">

## 4. حسابُ الطيفِ التردديِّ

نَستخدمُ طريقةَ ويلش \(Welch's method\) لتقديرِ كثافةِ الطاقةِ الطيفيّةِ. تُقسّمُ هذه الطريقةُ الإشارةَ إلى أجزاءَ متداخلةٍ، وتَحسبُ الطيفَ لكلِّ جزءٍ ثمَّ تُوسطّه، ما يُعطي تقديراً أنعمَ وأقلَّ تذبذباً من تحويلِ فورييه المباشر.

</div>

In [ ]:
from scipy import signal

freqs, psd = signal.welch(channel_data, fs=fs, nperseg=1024)
print(f'Frequency range: {freqs[0]:.1f} to {freqs[-1]:.1f} Hz')
print(f'Frequency resolution: {freqs[1]-freqs[0]:.2f} Hz')

<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ للنطاقاتِ التردديّةِ

يُظهرُ الرسمُ توزيعَ الطاقةِ على التردداتِ من 0 إلى 100 Hz، مع تلوينِ كلِّ نطاقٍ بلونٍ مُميّز.

**علامَ تُلاحظُ؟**
- الطاقةُ تَتركّزُ في النطاقاتِ المنخفضةِ (دلتا وثيتا وألفا)
- الطاقةُ تَتناقصُ مع زيادةِ الترددِ (قانونٌ طيفيٌّ $1/f$)
- قد تَظهرُ ذروةٌ في نطاقِ ألفا حولَ 10 Hz إذا كانَ المُشاركُ في حالةِ استرخاءٍ
- تَظهرُ ذروةٌ عندَ 50 Hz في بعضِ التسجيلاتِ بسببِ تدخّلِ شبكةِ الكهرباءِ

</div>

In [ ]:
import plotly.graph_objects as go

bands = {
    'Delta (0.5-4 Hz)':   (0.5, 4, 'purple'),
    'Theta (4-8 Hz)':     (4, 8, 'blue'),
    'Alpha (8-13 Hz)':    (8, 13, 'green'),
    'Beta (13-30 Hz)':    (13, 30, 'orange'),
    'Gamma (30-100 Hz)':  (30, 100, 'red'),
}

fig = go.Figure()
fig.add_trace(go.Scatter(x=freqs, y=psd, mode='lines',
                         name='PSD', line=dict(color='black', width=1)))

for name, (lo, hi, color) in bands.items():
    mask = (freqs >= lo) & (freqs <= hi)
    fig.add_trace(go.Scatter(x=freqs[mask], y=psd[mask], mode='lines',
                             name=name, line=dict(color=color, width=2),
                             fill='tozeroy', opacity=0.3))

fig.update_layout(height=500, title='EEG Frequency Bands (P4, subject 1)',
                  xaxis_title='Frequency (Hz)', yaxis_title='Power (uV^2/Hz)',
                  yaxis_type='log', xaxis_range=[0, 100])
fig.show()

<div dir="rtl" align="right">

## 6. خلاصةٌ

- تَحتوي إشارةُ EEG على خمسةِ نطاقاتٍ تردديّةٍ رئيسةٍ، كلٌّ منها يَرتبطُ بحالةٍ دماغيّةٍ
- تَتركّزُ الطاقةُ في النطاقاتِ المنخفضةِ وتَتناقصُ مع الترددِ وفقَ قانونٍ طيفيٍّ $1/f$
- طريقةُ ويلش تُعطي تقديراً أنعمَ للطيفِ من تحويلِ فورييه المباشر
- هذا التحليلُ هو الخطوةُ الأولى لفهمِ ما تَحملُهُ الإشارةُ من معلوماتٍ قبلَ تطبيقِ المرشّحاتِ في الفصلِ الرابعِ

</div>